# 3D materials

After exploring a 1D (polyacetylene) and a 2D system ($\mathrm{MoS}_2$), today we will look at iron, a typical 3D material. Iron is a good metal: it is magnetic (~2 $\mu_\mathrm{B}$ / atom) and it conducts electricity well. 

We will start by comparing BCC and FCC iron. Then we will look into surfaces and phonons.  

## 1. A tiny bit of magnetism

**1.** Load the functions that will be used. 

In [ ]:
"""
Remember, if you change something here you 
need to restart the kernel and execute
this cell again to import the changes. 
"""
from qe_funs import *

**2.** Compare FCC (```fcc.scf.in```) and BCC (```bcc.scf.in```) iron. Does the result make sense? 

Note that we are using direct coordinates now. 

In [ ]:
!pw.x < fcc.scf.in > fcc.scf.out
!grep '!' fcc.scf.out
!pw.x < bcc.scf.in > bcc.scf.out
!grep '!' bcc.scf.out

**3.** But we haven't included magnetism! Try these calculations again, starting from a guess magnetisation of 2.5 $\mu_{\mathrm{B}}$ / atom:

```
   nspin       = 2,            ! spin-polarized calculation
   starting_magnetization(1) = 2.5, ! initial magnetization for Fe
```

Is iron magnetic at the PBE level? 

In [ ]:
# You can do it!

**4.** Now let us look at the antiferromagnetic (AFM) configuration. Now the Fe atoms are no longer equivalent, so we have to do a little trick. Change ```ntyp```, add a ```starting_magnetisation(2)```, and rename iron atoms to ```Fe1``` and ```Fe2```. What is the energy of AFM BCC iron?

In [ ]:
# I believe in you!

**5.** Now that we have the energy of FM and AFM configurations, we can approximately determine the exchange coupling $J$ using the formula below. Recent [literature](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.116.217202) suggests $J \approx 1 \ \mathrm{mRy}$. What value do you get?

Note: it is generally considered a success if DFT gives the correct _sign_ of $J$.  

$$ J=\frac{E_{\mathrm{LS}}-E_{\mathrm{HS}}}
{S_\mathrm{HS}(S_\mathrm{HS}+1)} $$


## 2. Phonons

### 2.1. Lattice parameter optimisation

So far we have been focussed on the electronic structure of materials. Now we shall us look at the vibrational structure. Here the **phonons**, collective vibrations of atoms in the lattice, play a major role, determining properties such as the bulk modulus, thermal conductivity and speed of sound. Phonons also affect electrical conductivity by modifying the relaxation time $\tau$, which may also be computed using QE (probably not in 3 hours, though). 

**6.** To start, inspect  ```scf.in.template``` and ```test_lat_par.sh```. Then, determine the optimal lattice parameter $a$ and plot the equation of state ($E$ vs $a$) for iron using the code below. 

<span style="color:red">Note 1: If calculations are taking longer than ~25 seconds, reduce the planewave cutoffs to 15 and 150 Ry, respectively. If that's still too slow, reduce the **k**-grid to 6 6 6 and increase smearing (```degauss```) to 0.03 Ry.</span>

Note 2: We are making serious compromises in the $\bf{k}$-grid and plane wave cutoff to fit the calculations in workshop format. These results will not be nice. 

Note 3: We could also use the ```'vc-relax'``` algorithm for lattice relaxation, but this is less accurate for technical reasons. For a complicated (heterostructure) material, this would probably be a more straightforward approach. 

In [ ]:
# !bash test_lat_par.sh

data = np.genfromtxt("energy_lat_par.dat")
data = data[data[:, 0].argsort()]
a = data[:, 0]
E = data[:, 1]
plt.plot(a, E, "o--", color=colours["orange"])
plt.xlabel(r"$a$ ($\mathrm{\AA}$)")
plt.ylabel("$E$ (Ry)")

This equation-of-state plot could be used to compute the bulk modulus, which is defined as the derivative of pressure with respect to volume: 

$$ K = -V\frac{dP}{dV} $$

Recalling that 

$$ P = -\frac{dE}{dV} $$

at equilibrium $a = a_0$ we obtain 

$$
K_0 = \frac{1}{9a_0}\left.\frac{d^2E}{da^2}\right|_{a_0}
$$

If we fit $E(a)$ near its minimum, we can estimate $K_0$ directly from the curvature. We could even repurpose the effective mass code from week 3 to do it. 

**7.** Derive the formula for $K_0$ above! 

**8.** Compare the (unconverged) DFT-predicted density of iron with its experimental value ($7.87 \ \mathrm{g/cm}^3$). 

### 2.2. Surfaces

**9.** By this point you might be bored with 3D solids. Fortunately we can make surfaces, and then put something on them! Let us try to put benzene on an iron (110) surface. First, insert your optimal lattice parameter into ```scf.in```, and then use the ```surface``` function from ASE to make a (110) surface slab with 5 layers and 10 $\mathrm{\AA}$ of vacuum in the z-direction.  

In [ ]:
from ase.build import surface

filename = "scf.in"

atoms = read(f"{filename}", format='espresso-in')
# slab = surface(# your keywords here) 
write(f"surface.xsf", slab) 

**10.** We can try to put a molecule of benzene on our surface. Practically we will first start the phonon calculations in **11.** and then continue here while that calculation is running. A reasonable workflow might look like this: 
 - draw benzene using Avogadro and note its dimensions
 - use the transformation matrix in Vesta to create a supercell of appropriate size 
 - export the supercell both as a cartesian ```POSCAR``` and as a ```surface.xyz``` file
 - open ```surface.xyz``` using Avogadro and place benzene in the desired position, save as ```adsorbed.xyz```
 - paste the benzene coordinates from ```adsorbed.xyz``` into ```POSCAR```
 - ??? 
 - publish

We will skip the two final steps as they might take about 6 months. 

### 2.3. Phonon calculations

In Quantum Espresso, phonon calculations are done by computing the derivative of the density with respect to position. This is done at several $\bf{q}$-points in reciprocal space. At $\bf{q} = (0, 0, 0)$ atomic displacements are the same across all unit cells, but at $\bf{q} \neq (0, 0, 0)$ this is not the case. Here we will be using a minimal 2x2x2 $\bf{q}$-grid to save time; for production-level calculations one would need a much denser grid.

**11.** Inspect ```phonon.in``` and run the code below. This might take 15-20 min.  

In [ ]:
!pw.x < scf.in > scf.out
!ph.x < phonon.in > phonon.out # this will take a while
!q2r.x < q2r.in > q2r.out

**12.** To visualise the results we can plot the phonon band structure. This is done by interpolation, which works well when you have a good $\bf{q}$-grid, so here it will only give qualitative results. First, head over to https://seekpath.materialscloud.io and identify an appropriate high-symmetry path for iron. Then, inspect the ```matdyn_bands.in``` file and run the code below. 

In [ ]:
!matdyn.x < matdyn_bands.in > matdyn_bands.out

**13.** Now inspect ```Fe.freq.gp```. The first column corresponds to energies, while the next six correspond to the individual phonon bands. Complete the code to plot them. Which bands are acoustic and which are optical? 

Note: a nice tool is available at https://interactivephonon.materialscloud.io. 

In [ ]:
bands = np.genfromtxt('Fe.freq.gp')
spacing = 40
plt.plot(bands[:, 0], bands[:, 1], "-", color=colours["blue"])
# ... and so on for the other bands
plt.axvline(bands[spacing, 0], color="k", linestyle="--") # etc
plt.xticks([0, bands[spacing, 0]], ["Γ", "H"]) # etc
plt.xlabel(r"$\bf{q}$")
plt.ylabel(r"$\tilde{\nu}$ (cm$^{-1}$)")
plt.xlim(0, bands[-1, 0]) 
plt.ylim(0, 300)

**14.** We can determine the speed of sound in iron by looking at the slope of the acoustic phonon bands close to the $\Gamma$ point: 

$$ v_s = \left.\frac{d\omega}{dq}\right|_{q \to 0} $$

Close to $\Gamma$ the optical bands tend to be linear, so this can be done from the first few points in ```Fe.freq.gp```.

The experimental value is 5000-6000 m/s. What do you get?

In [ ]:
delta_q_reduced = bands[1, 0] # this is in reduced coordinates
delta_nu_cm_inv = bands[1, 1] # this is in cm^-1

# ... velocity is in m/s 

**15.** Finally, we can determine the phonon density of states (DOS), which is especially important for thermal conductance $\kappa$: 

$$ \kappa = \frac{1}{3} \int C(\omega, T) \, v^2(\omega) \, \tau(\omega) \, g(\omega) \, d\omega $$

where $C(\omega, T)$ is the phonon heat capacity, $v(\omega)$ is the group velocity, $\tau(\omega)$ is the relaxation time, and $g(\omega)$ is the phonon DOS. All quantities in this equation can be obtained using DFT, although in practice some of them are usually approximated or taken from experiment. 

To compute the phonon DOS, run the code below and then inspect ```Fe.dos```.

In [ ]:
!matdyn.x < matdyn_dos.in > matdyn_dos.out
# plotting the DOS is left as an exercise for the reader :)

## 3. Electronic band structure and DOS revision

**16.** Finally, revise what you have learned so far and plot the electronic band structure and DOS on your own. 

In [ ]:
# !pw.x < scf.in > scf_morek.out
# !pw.x < nscf_dos.in > nscf_dos.out
# !dos.x < dos.in > dos.out
# !pw.x < nscf_bands.in > nscf_bands.out
# !bands.x < bands_up.in > bands_up.out
# !bands.x < bands_dn.in > bands_dn.out